# 💳 Credit Card Fraud Detection with Deep Learning

**Author:** Perzy  
**Stack:** TensorFlow/Keras, Scikit-learn, Pandas, Seaborn  
**Goal:** Build a production-grade fraud detection system that handles extreme class imbalance (99.83% legit vs 0.17% fraud)

---

## Table of Contents
1. [Setup & Data Loading](#1)
2. [Exploratory Data Analysis](#2)
3. [Data Preprocessing](#3)
4. [Model Architecture](#4)
5. [Training](#5)
6. [Evaluation & Results](#6)
7. [Autoencoder Anomaly Detection](#7)
8. [Threshold Optimization](#8)
9. [Conclusions](#9)

<a id='1'></a>
## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import warnings
warnings.filterwarnings('ignore')

# Local modules
import sys
sys.path.append('..')
from src.preprocessing import (
    generate_synthetic_data, preprocess, split_data, apply_smote, scale_features
)
from src.model import (
    build_classifier, build_autoencoder, train_classifier,
    train_autoencoder, evaluate_model, find_optimal_threshold,
    get_class_weights
)
from src.visualize import (
    plot_class_distribution, plot_amount_distribution,
    plot_time_distribution, plot_correlation_matrix,
    plot_top_feature_correlations, plot_training_history,
    plot_confusion_matrix, plot_roc_and_pr_curves,
    plot_threshold_analysis
)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Option 1: Load real Kaggle dataset
# df = pd.read_csv('data/creditcard.csv')

# Option 2: Generate synthetic data (same structure as Kaggle dataset)
# The real dataset has 284,807 transactions with 492 frauds (0.172%)
# Features V1-V28 are PCA-transformed for confidentiality
df = generate_synthetic_data(n_samples=284807, fraud_ratio=0.00173)
df.head()

In [ ]:
print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['Class'].value_counts())
print(f'\nFraud percentage: {df["Class"].mean():.4%}')
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'\nBasic statistics:')
df.describe()

<a id='2'></a>
## 2. Exploratory Data Analysis

Understanding the data distribution is critical when dealing with fraud detection — the extreme class imbalance means standard accuracy is meaningless.

In [ ]:
# Class imbalance visualization
plot_class_distribution(df, save_path='../visualizations/class_distribution.png')

In [ ]:
# Transaction amount analysis
plot_amount_distribution(df, save_path='../visualizations/amount_distribution.png')

In [ ]:
# Temporal patterns
plot_time_distribution(df, save_path='../visualizations/time_distribution.png')

In [ ]:
# Feature correlations
plot_correlation_matrix(df, save_path='../visualizations/correlation_matrix.png')

In [ ]:
# Most predictive features
plot_top_feature_correlations(df, n=15, save_path='../visualizations/top_correlations.png')

In [ ]:
# Deeper look at the top correlated features
top_features = ['V14', 'V10', 'V12', 'V17', 'V4', 'V11', 'V3']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, feat in enumerate(top_features):
    ax = axes[idx // 4, idx % 4]
    df[df['Class'] == 0][feat].hist(bins=50, ax=ax, alpha=0.6, color='#2ecc71', label='Legit', density=True)
    df[df['Class'] == 1][feat].hist(bins=50, ax=ax, alpha=0.7, color='#e74c3c', label='Fraud', density=True)
    ax.set_title(feat, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)

axes[1, 3].axis('off')
plt.suptitle('Distribution of Top Features by Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### Key EDA Findings
- **Extreme imbalance**: Only 0.17% of transactions are fraudulent — a naive model predicting all-legit gets 99.83% accuracy
- **Amount patterns**: Fraudulent transactions tend to have different amount distributions
- **Feature separation**: V14, V10, V12, V17 show clear separation between classes
- **No missing values**: Dataset is clean and ready for modeling

<a id='3'></a>
## 3. Data Preprocessing

Key steps:
1. Scale `Amount` and `Time` features (V1-V28 are already PCA-transformed)
2. Stratified train/val/test split (70/10/20)
3. Feature standardization
4. SMOTE oversampling for training data

In [ ]:
# Scale Amount and Time
df_processed = preprocess(df, scale_amount=True, scale_time=True)
print(f'Processed shape: {df_processed.shape}')
df_processed.head()

In [ ]:
# Stratified split
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    df_processed, test_size=0.2, val_size=0.1, seed=SEED
)

In [ ]:
# Standardize features
X_train_scaled, X_val_scaled, X_test_scaled, scaler = scale_features(X_train, X_val, X_test)
print(f'Feature dimension: {X_train_scaled.shape[1]}')

In [ ]:
# Apply SMOTE to training data only
X_train_resampled, y_train_resampled = apply_smote(
    X_train_scaled, y_train, sampling_strategy=0.1, seed=SEED
)

<a id='4'></a>
## 4. Model Architecture

### Approach 1: Deep Neural Network Classifier
A multi-layer neural network with:
- Batch normalization for training stability
- Dropout regularization (30%) to prevent overfitting
- L2 weight regularization
- Class weights to handle imbalance

### Approach 2: Autoencoder for Anomaly Detection
- Trained only on legitimate transactions
- Fraudulent transactions produce higher reconstruction error
- No labels needed — unsupervised approach

In [ ]:
# Build the classifier
input_dim = X_train_scaled.shape[1]
classifier = build_classifier(input_dim=input_dim, dropout_rate=0.3)
classifier.summary()

<a id='5'></a>
## 5. Training

Training strategy:
- **Class weights** inversely proportional to class frequency
- **Early stopping** with patience=10 monitoring val_loss
- **Learning rate reduction** on plateau
- **Large batch size** (2048) for stable gradient estimates on imbalanced data

In [ ]:
# Train with class weights on SMOTE-resampled data
history = train_classifier(
    classifier,
    X_train_resampled, y_train_resampled,
    X_val_scaled, y_val,
    epochs=100,
    batch_size=2048,
    use_class_weights=True
)

In [ ]:
# Training curves
plot_training_history(history, save_path='../visualizations/training_history.png')

<a id='6'></a>
## 6. Evaluation & Results

In [ ]:
# Evaluate on test set
results = evaluate_model(classifier, X_test_scaled, y_test, threshold=0.5)

In [ ]:
# Confusion matrix
plot_confusion_matrix(y_test, results['y_pred'],
                      save_path='../visualizations/confusion_matrix.png')

In [ ]:
# ROC and PR curves
plot_roc_and_pr_curves(y_test, results['y_pred_proba'],
                       save_path='../visualizations/roc_pr_curves.png')

<a id='7'></a>
## 7. Autoencoder Anomaly Detection

An alternative unsupervised approach: train the autoencoder to reconstruct legitimate transactions. Fraud shows up as high reconstruction error.

In [ ]:
# Build autoencoder
autoencoder, encoder = build_autoencoder(input_dim=input_dim, encoding_dim=14)
autoencoder.summary()

In [ ]:
# Train on legitimate transactions only
X_train_legit = X_train_scaled[y_train == 0]
print(f'Training autoencoder on {X_train_legit.shape[0]:,} legitimate transactions')

ae_history = train_autoencoder(
    autoencoder, X_train_legit,
    X_val_scaled, y_val,
    epochs=100, batch_size=512
)

In [ ]:
# Compute reconstruction error
X_test_reconstructed = autoencoder.predict(X_test_scaled, verbose=0)
reconstruction_error = np.mean(np.square(X_test_scaled - X_test_reconstructed), axis=1)

# Visualize reconstruction error distribution
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(reconstruction_error[y_test == 0], bins=100, alpha=0.6, color='#2ecc71',
        label='Legitimate', density=True)
ax.hist(reconstruction_error[y_test == 1], bins=50, alpha=0.7, color='#e74c3c',
        label='Fraud', density=True)
ax.set_title('Autoencoder Reconstruction Error Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Mean Squared Error')
ax.set_ylabel('Density')
ax.legend(fontsize=12)
ax.set_xlim(0, np.percentile(reconstruction_error, 99))
plt.tight_layout()
plt.savefig('../visualizations/reconstruction_error.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean reconstruction error (Legit):  {reconstruction_error[y_test == 0].mean():.6f}')
print(f'Mean reconstruction error (Fraud):  {reconstruction_error[y_test == 1].mean():.6f}')
print(f'Ratio: {reconstruction_error[y_test == 1].mean() / reconstruction_error[y_test == 0].mean():.1f}x higher for fraud')

<a id='8'></a>
## 8. Threshold Optimization

The default 0.5 threshold isn't optimal for imbalanced data. Let's find the threshold that maximizes F1-score.

In [ ]:
# Find optimal threshold
optimal_threshold = plot_threshold_analysis(
    y_test, results['y_pred_proba'],
    save_path='../visualizations/threshold_analysis.png'
)
print(f'\nOptimal threshold: {optimal_threshold}')

In [ ]:
# Re-evaluate with optimal threshold
print(f'\n{"="*60}')
print(f'RESULTS WITH OPTIMAL THRESHOLD ({optimal_threshold:.2f})')
print(f'{"="*60}')
optimized_results = evaluate_model(classifier, X_test_scaled, y_test, threshold=optimal_threshold)

In [ ]:
# Final confusion matrix with optimal threshold
plot_confusion_matrix(y_test, optimized_results['y_pred'],
                      save_path='../visualizations/confusion_matrix_optimized.png')

<a id='9'></a>
## 9. Conclusions

### Results Summary

| Metric | Default (0.5) | Optimized |
|--------|--------------|----------|
| ROC-AUC | See above | Same (threshold-independent) |
| PR-AUC | See above | Same (threshold-independent) |
| F1-Score | See above | Improved |

### Key Takeaways

1. **Class imbalance is the core challenge** — vanilla accuracy is meaningless at 99.83% legit. PR-AUC and F1 are the right metrics.

2. **SMOTE + class weights** together outperform either alone — SMOTE generates synthetic minority samples while class weights adjust the loss function.

3. **Autoencoder anomaly detection** provides a complementary unsupervised signal — fraud transactions have significantly higher reconstruction error, which could be used as an ensemble feature.

4. **Threshold optimization matters** — moving from the default 0.5 to the F1-optimal threshold meaningfully improves operational performance.

### Production Considerations
- **Real-time scoring**: The trained model inference runs in <1ms per transaction
- **Monitoring**: Track feature drift and model degradation over time
- **Ensemble**: Combine classifier + autoencoder scores for more robust detection
- **Cost-sensitive**: In production, weight FN (missed fraud) much higher than FP (false alarm)